In [ ]:
#| default_exp security

# Security

> AWS security baseline — CloudTrail, GuardDuty, Config, Security Hub, IAM password policy, VPC flow logs.

In [ ]:
#| export
import json

## Security Baseline

Enable AWS-native security controls with one call each. All functions are idempotent.

```python
from awseasy.security import (
    enable_cloudtrail, enable_guardduty, enable_config,
    enable_security_hub, set_iam_password_policy, enable_vpc_flow_logs,
)

# Stand-alone
enable_cloudtrail(auth, 'my-trail', 'my-log-bucket')
enable_guardduty(auth)
set_iam_password_policy(auth)

# Or via GenAIStack
stack = GenAIStack(auth, 'myapp', compliance=SOC2)
stack.provision(security_baseline=True)
```

In [ ]:
#| export
def _ct(auth):
    return auth.session.client('cloudtrail')

def enable_cloudtrail(auth, name, bucket, kms_key_id=None,
                      multi_region=True, log_validation=True) -> dict:
    'Enable CloudTrail — multi-region, log file validation, S3 delivery. Idempotent.'
    client = _ct(auth)
    kwargs = dict(
        Name=name,
        S3BucketName=bucket,
        IsMultiRegionTrail=multi_region,
        EnableLogFileValidation=log_validation,
        IncludeGlobalServiceEvents=True,
    )
    if kms_key_id:
        kwargs['KMSKeyId'] = kms_key_id
    try:
        trail = client.create_trail(**kwargs)
        client.start_logging(Name=name)
        return trail
    except client.exceptions.TrailAlreadyExistsException:
        client.start_logging(Name=name)
        return client.get_trail(Name=name)['Trail']


## Amazon GuardDuty

Threat detection — network, IAM, S3 activity. Serverless, enable once per region.

In [ ]:
#| export
def _gd(auth):
    return auth.session.client('guardduty')

def enable_guardduty(auth) -> dict:
    'Enable GuardDuty in the current region. Idempotent — returns existing detector if already enabled.'
    client = _gd(auth)
    detectors = client.list_detectors()['DetectorIds']
    if detectors:
        return client.get_detector(DetectorId=detectors[0])
    return client.create_detector(Enable=True,
                                   FindingPublishingFrequency='SIX_HOURS')


## AWS Config

Continuous resource configuration recording with S3 delivery.

In [ ]:
#| export
def _cfg(auth):
    return auth.session.client('config')

def enable_config(auth, bucket, role_arn) -> dict:
    'Enable AWS Config recorder + delivery channel. Idempotent.'
    client = _cfg(auth)
    # Configuration recorder
    client.put_configuration_recorder(
        ConfigurationRecorder={
            'name': 'default',
            'roleARN': role_arn,
            'recordingGroup': {'allSupported': True,
                               'includeGlobalResourceTypes': True},
        })
    # Delivery channel
    client.put_delivery_channel(
        DeliveryChannel={
            'name': 'default',
            's3BucketName': bucket,
            'configSnapshotDeliveryProperties': {'deliveryFrequency': 'TwentyFour_Hours'},
        })
    client.start_configuration_recorder(ConfigurationRecorderName='default')
    return client.describe_configuration_recorders()['ConfigurationRecorders'][0]


## AWS Security Hub

Aggregates findings from GuardDuty, Config, Inspector, and Macie. Enables CIS AWS Foundations benchmark.

In [ ]:
#| export
def _sh(auth):
    return auth.session.client('securityhub')

def enable_security_hub(auth, cis_standard=True) -> dict:
    'Enable Security Hub + CIS AWS Foundations standard. Idempotent.'
    client = _sh(auth)
    try:
        result = client.enable_security_hub(EnableDefaultStandards=True)
    except client.exceptions.ResourceConflictException:
        result = client.describe_hub()
    if cis_standard:
        cis_arn = (f'arn:aws:securityhub:{auth.region}::standards/'
                   'cis-aws-foundations-benchmark/v/1.4.0')
        try:
            client.batch_enable_standards(
                StandardsSubscriptionRequests=[{'StandardsArn': cis_arn}])
        except client.exceptions.ResourceConflictException:
            pass  # already subscribed
    return result


## IAM Password Policy

Enforce strong password requirements across all IAM users.

In [ ]:
#| export
def set_iam_password_policy(auth, min_length=14, require_symbols=True,
                             require_numbers=True, require_uppercase=True,
                             require_lowercase=True, max_age=90,
                             reuse_prevention=24, hard_expiry=False) -> dict:
    'Set account-wide IAM password policy. Defaults match CIS AWS Foundations benchmark.'
    iam = auth.session.client('iam')
    iam.update_account_password_policy(
        MinimumPasswordLength=min_length,
        RequireSymbols=require_symbols,
        RequireNumbers=require_numbers,
        RequireUppercaseCharacters=require_uppercase,
        RequireLowercaseCharacters=require_lowercase,
        MaxPasswordAge=max_age,
        PasswordReusePrevention=reuse_prevention,
        HardExpiry=hard_expiry,
        AllowUsersToChangePassword=True,
    )
    return iam.get_account_password_policy()['PasswordPolicy']


## VPC Flow Logs

Capture VPC network traffic metadata to CloudWatch Logs.

In [ ]:
#| export
def enable_vpc_flow_logs(auth, vpc_id, log_group='/aws/vpc/flowlogs',
                          traffic_type='ALL') -> dict:
    'Enable VPC flow logs to CloudWatch Logs. Idempotent — returns existing flow log if already configured.'
    ec2 = auth.session.client('ec2')
    logs = auth.session.client('logs')
    iam = auth.session.client('iam')

    # Ensure log group exists
    try:
        logs.create_log_group(logGroupName=log_group)
    except logs.exceptions.ResourceAlreadyExistsException:
        pass

    # Check existing
    existing = ec2.describe_flow_logs(
        Filters=[{'Name': 'resource-id', 'Values': [vpc_id]},
                 {'Name': 'traffic-type', 'Values': [traffic_type]}])['FlowLogs']
    if existing:
        return existing[0]

    # IAM role for flow logs
    trust = json.dumps({'Version': '2012-10-17', 'Statement': [{
        'Effect': 'Allow',
        'Principal': {'Service': 'vpc-flow-logs.amazonaws.com'},
        'Action': 'sts:AssumeRole',
    }]})
    role_name = 'awseasy-vpc-flow-logs-role'
    try:
        role = iam.create_role(RoleName=role_name,
                               AssumeRolePolicyDocument=trust)['Role']
        iam.put_role_policy(
            RoleName=role_name,
            PolicyName='vpc-flow-logs',
            PolicyDocument=json.dumps({'Version': '2012-10-17', 'Statement': [{
                'Effect': 'Allow',
                'Action': ['logs:CreateLogGroup', 'logs:CreateLogStream',
                           'logs:PutLogEvents', 'logs:DescribeLogGroups',
                           'logs:DescribeLogStreams'],
                'Resource': '*',
            }]}),
        )
    except iam.exceptions.EntityAlreadyExistsException:
        role = iam.get_role(RoleName=role_name)['Role']

    result = ec2.create_flow_logs(
        ResourceIds=[vpc_id],
        ResourceType='VPC',
        TrafficType=traffic_type,
        LogDestinationType='cloud-watch-logs',
        LogGroupName=log_group,
        DeliverLogsPermissionArn=role['Arn'],
    )
    return result
